In [2]:
import math
import random

import torch
import numpy as np
import torch.nn as nn
from collections import Counter
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
set_seed()

image_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
])
train_full = datasets.CIFAR10(root="data", train=True, download=True, transform=image_transform)
test_full = datasets.CIFAR10(root="data", train=False, download=True, transform=image_transform)

TRAIN_LIMIT = 4096
TEST_LIMIT = 1000

train_dataset = Subset(train_full, range(TRAIN_LIMIT))
test_dataset = Subset(test_full, range(TEST_LIMIT))

labels = [test_full.targets[i] for i in test_dataset.indices]
class_counts = Counter(labels)

print(class_counts)
print(type(class_counts))

class_names = test_full.classes
for class_idx, count in sorted(class_counts.items()):
    print(f"{class_names[class_idx]}: {count}")

print("Min:", min(class_counts.values()), "Max:", max(class_counts.values()))

Device: cpu
Counter({6: 112, 9: 109, 8: 106, 3: 103, 0: 103, 7: 102, 2: 100, 4: 90, 1: 89, 5: 86})
<class 'collections.Counter'>
airplane: 103
automobile: 89
bird: 100
cat: 103
deer: 90
dog: 86
frog: 112
horse: 102
ship: 106
truck: 109
Min: 86 Max: 112


In [3]:
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False, num_workers=0)

class_names = train_full.classes
print(f"classes: {class_names}")

images, labels = next(iter(train_loader))
print(f"images shape: {images.shape}")
print(f"labels shape: {labels.shape}")

classes: ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']
images shape: torch.Size([64, 3, 32, 32])
labels shape: torch.Size([64])


Question 0.1:

<div style="direction: rtl; text-algin: right;">
generalization بهتر، فدای زمان پردازش می‌شود
</div>

Question A1:
<div style="direction: rtl; text-algin: right;">
با توجه به چند کلاسه بودن مسئله، استفاده از cross-entropy گزینه مناسب تری است
</div>

In [4]:
class ToyModel(nn.Module):
    def __init__(self):
        super(ToyModel, self).__init__()
        self.network =  nn.Sequential(
            nn.Flatten(),
            nn.Linear(32 * 32 * 3, 10)
        )
        # self.network = nn.Sequential(nn.Linear(32 * 32 * 3, 10), nn.ReLU())

    def forward(self, x):
        return self.network(x)

In [5]:
toy_model = ToyModel().to(device=DEVICE)

logits = toy_model(images.to(device=DEVICE))
print(logits.shape)
print(logits[0].argmax())
print(labels[0])


# 64 = Batch size 10 = N(classes)

# If Batch size was 32 the first dimension will be 32 and the second dimension will be 10

torch.Size([64, 10])
tensor(8)
tensor(0)


In [6]:
# calculate loss with torch
criterion = nn.CrossEntropyLoss()
loss = criterion(logits, labels)
print(f"loss torch {loss}")
# By hand
probabilites = torch.softmax(logits, dim=1).detach().numpy()

loss_manual = np.mean(-(np.log(probabilites[torch.arange(len(labels)),labels])))
print(f"loss torch {loss_manual}")
print(f"log 1/10 : {- np.log(0.1)}")
print(f"log 1/100 : {- np.log(0.01)}")
print(f"log 9/10 : {- np.log(0.9)}")

# هر وقت مدل با اطمینان زیاد اشتباه پیش بینی کند یا احتمال کمی برای کلاس درست در نظر بگیرد لاس جریمه بزرگی برای آن در نظر میگیرد

loss torch 2.3087856769561768
loss torch 2.3087856769561768
log 1/10 : 2.3025850929940455
log 1/100 : 4.605170185988091
log 9/10 : 0.10536051565782628


# non-Safety

In [7]:
# X
generator = torch.Generator().manual_seed(42)

logit = torch.randn((100,10),dtype=torch.float,generator=generator).to(device=DEVICE)*50
target = torch.zeros(size=(100,),dtype=torch.long).to(DEVICE)

P = torch.exp(logit)/torch.sum(torch.exp(logit), dim=1, keepdim=True)

loss = torch.mean(-torch.log(P))
print(f"loss torch {loss}")
print(f"loss torch one row {-torch.log(P)[10]}\n-----------------")


print(f"Controled loss with nn.CrossEntropyLoss() = {criterion(logit, target)}")

loss torch nan
loss torch one row tensor([3.1621e+01, 3.4496e+01, 1.0312e+01, 8.6166e+01, 7.2922e+01, 2.6021e+01,
        4.3421e+01, 5.4735e+01, 5.0413e+01, 3.3260e-05])
-----------------
Controled loss with nn.CrossEntropyLoss() = 87.07770538330078


In [8]:
generator = torch.Generator().manual_seed(42)
logit = torch.randn((100,10),dtype=torch.float,generator=generator).to(device=DEVICE)*50
target = torch.zeros(size=(100,),dtype=torch.long).to(DEVICE)

# maximum logits
m = torch.max(logit, dim=1, keepdim=True).values


target_logits = logit[torch.arange(len(target)), target].unsqueeze(1)
term2 =m + torch.log(torch.sum(torch.exp(logit-m),dim=1 ,keepdim=True))
loss= -( target_logits - term2)
print(f"calculate logexpsum by hand : {torch.mean(loss)}")
print(f"by torch {criterion(logit, target)}")
func_loss = - (target_logits - (m + torch.logsumexp(logit-m, dim=1)))
# loss = torch.logsumexp(logit, dim=1)
print(f"with logsum torch : {torch.mean(func_loss)}")

calculate logexpsum by hand : 87.07771301269531
by torch 87.07770538330078
with logsum torch : 87.07771301269531


In [13]:
class_names

['airplane',
 'automobile',
 'bird',
 'cat',
 'deer',
 'dog',
 'frog',
 'horse',
 'ship',
 'truck']